# Notebook 01 — Curated Thesis Corpus

**Job of this notebook:** turn the raw TMDB extract into the film corpora every later notebook reads. It owns *every* decision about which films are in the thesis.

### Design decisions
| # | Decision | Why it matters |
|---|----------|----------------|
| 1 | **Robust cast/list parser.** Names with apostrophes or nicknames (`Peter O'Toole, Conan O'Brien`) are parsed correctly; a naive comma split turns them into `Toole, Conan O`. | `cast_clean` feeds the star-power measure in NB05. The diagnostics report how many rows a naive split would corrupt. |
| 2 | **Non-story event filter with review.** Keyword rules flag candidate event recordings; a candidate is removed only if it is a recording without a plot: wrestling/MMA event broadcasts, concert films, stand-up specials (172 films). Keyword rules alone would remove 568 films, 396 of them real story films (Terminator 2, Godzilla, Hellboy, American Gangster …) caught by words like *vengeance*, *tournament*, *ceremony*, or by `tna` matching inside "Vie**tna**m". | `df_network_corpus.csv`: **19,919 films**. Every decision is logged. `CFG['event_filter'] = "legacy"` gives the keyword-only corpus (19,523 films) for comparison. |
| 3 | **Fixed snapshot date** instead of today's date. | Funnel counts do not change with the day the notebook runs. |
| 4 | **Runtime step split** into "missing/zero runtime" and "outside 70–240 min". | A clearer funnel figure. |
| 5 | **Safety guard.** Before writing, the corpus is compared with the files already on Drive and with the embedding index NB03 used. If films or their order would change, nothing is overwritten. | Protects community IDs and manual labels. |
| 6 | **Diagnostics** for sample decisions that need a justification: documentaries as secondary genre, implausible budgets, non-English overviews, quarantine false positives. | Reported only — no data changes unless a flag is switched on. |
| 7 | One `CFG`, a safe Drive mount, vectorised parsing and a `corpus_manifest.json` fingerprint that later notebooks check. | Reproducibility. |

Optional changes are behind flags in `CFG['options']`, all off.

In [14]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 1 — CONFIG  (the only cell that needs editing)
# ═══════════════════════════════════════════════════════════════════════════
CFG = {
    "project_dir": "/content/drive/MyDrive/thesis_movie_project",
    "raw_filename": "TMDB_all_movies.csv",
    "local_cache_dir": "/content/thesis_movie_project/raw",   # fast local copy of the raw CSV

    # ── sample definition ───────────────────────────────────────────────────
    "year_min": 1990,
    "year_max": 2024,
    "snapshot_date": "2024-12-31",      # replaces Timestamp.today(); must be >= year_max end
    "runtime_min": 70,
    "runtime_max": 240,
    "overview_words_thesis": 25,
    "overview_words_semantic": 35,
    "vote_count_thesis": 1,
    "vote_count_semantic": 25,
    "vote_count_display": 50,
    "exclude_tv_movie": True,
    "exclude_documentary_primary": True,
    "cast_top_n": 5,
    "roi_bounds": (0, 100),              # exclusive bounds on revenue/budget

    # ── optional methodological changes (all OFF = reproduce current corpus) ─
    "options": {
        "exclude_documentary_any_genre": False,   # also drop films with Documentary as a secondary genre
        "min_plausible_money": None,              # e.g. 10_000 → treat budget/revenue below this as missing
    },

    # ── non-story event filter ──────────────────────────────────────────────
    # "strict": remove only recordings with no plot (wrestling/MMA broadcasts, concert films,
    #           stand-up specials). Decided 2026-10-06 after auditing all 568 keyword-rule removals.
    # "legacy": keyword rules only (also removes 396 real story films); for comparison.
    "event_filter": "strict",

    # ── safety ──────────────────────────────────────────────────────────────
    # True because the strict event filter deliberately changes the network corpus.
    # After NB03 has rebuilt the network from the new corpus, set this back to False.
    "allow_corpus_change": False,
    "expected_network_rows": {"strict": 19919, "legacy": 19523},
}
print("CONFIG loaded.")


CONFIG loaded.


In [15]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 2 — Imports, Drive mount (idempotent), paths
# ═══════════════════════════════════════════════════════════════════════════
import os, re, json, shutil, hashlib, warnings
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)

try:
    display
except NameError:            # running outside Jupyter
    display = print

try:
    from google.colab import drive
    if Path("/content/drive/MyDrive").exists():
        print("Google Drive already mounted.")
    else:
        drive.mount("/content/drive")
except Exception as e:
    print(f"Drive mount skipped: {e}")

PROJECT_DIR        = Path(CFG["project_dir"])
RAW_PATH           = PROJECT_DIR / "raw" / CFG["raw_filename"]
LOCAL_RAW_PATH     = Path(CFG["local_cache_dir"]) / CFG["raw_filename"]
CORE_DIR           = PROJECT_DIR / "processed" / "core"
FEATURES_DIR       = PROJECT_DIR / "processed" / "features"
TABLES_DIR         = PROJECT_DIR / "outputs" / "tables"
AUDIT_DIR          = TABLES_DIR / "corpus_audit"
EMBED_INDEX_FILE   = FEATURES_DIR / "sbert_minilm_embedding_index.csv"   # written by NB03

for d in [CORE_DIR, FEATURES_DIR, TABLES_DIR, AUDIT_DIR, LOCAL_RAW_PATH.parent]:
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("Raw file exists:", RAW_PATH.exists())


Google Drive already mounted.
PROJECT_DIR: /content/drive/MyDrive/thesis_movie_project
Raw file exists: True


In [16]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 3 — Utilities (vectorised)
# ═══════════════════════════════════════════════════════════════════════════
NULL_TOKENS = {"", "nan", "none", "null", "[]", "unknown", "n/a"}

def clean_text(s: pd.Series) -> pd.Series:
    return s.fillna("").astype(str).str.replace(r"\s+", " ", regex=True).str.strip()

def to_num(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors="coerce")

def pick_col(columns, candidates, required=False):
    for c in candidates:
        if c in columns:
            return c
    if required:
        raise KeyError(f"None of these required columns were found: {candidates}")
    return None

def parse_list_column(s: pd.Series, top_n=None) -> pd.Series:
    """Vectorised list parser.

    Handles 'A, B, C' and "['A', 'B']". Unlike a naive comma split it does not break on names
    containing apostrophes or quotes (O'Toole, "The Rock")."""
    s = clean_text(s)
    listlike = s.str.startswith("[")
    stripped = (s.str.replace(r"^\[|\]$", "", regex=True)
                  .str.replace(r"(^|,\s*)['\"]|['\"](?=\s*,|$)", r"\1", regex=True))
    s = s.where(~listlike, stripped)
    parts = s.str.split(r"\s*,\s*", regex=True).explode().str.strip()
    parts = parts[~parts.str.lower().isin(NULL_TOKENS)]
    if top_n:
        parts = parts.groupby(level=0).head(top_n)
    joined = parts.groupby(level=0).agg(", ".join)
    return joined.reindex(s.index, fill_value="")

def legacy_parse(x, top_n=None):
    """Naive comma-split parser, kept only to measure how many rows it would corrupt."""
    if pd.isna(x): return ""
    s = str(x).strip()
    if s.lower() in {"", "nan", "none", "null", "[]"}: return ""
    q = re.findall(r"'([^']+)'|\"([^\"]+)\"", s)
    vals = [(a or b).strip() for a, b in q] if q else [p.strip(" []'\"") for p in s.split(",")]
    vals = [v for v in vals if v and v.lower() not in {"nan", "none", "null"}]
    return ", ".join(vals[:top_n] if top_n else vals)

def normalize_title(s: pd.Series) -> pd.Series:
    s = clean_text(s).str.lower().str.replace(r"[^a-z0-9\s]", " ", regex=True)
    return s.str.replace(r"\s+", " ", regex=True).str.strip()

def fingerprint(ids: pd.Series) -> str:
    """Order-sensitive hash of a film-id sequence. Same corpus + same order → same hash."""
    return hashlib.sha256(",".join(ids.astype(str)).encode()).hexdigest()[:16]

class Funnel:
    def __init__(self, n0):
        self.rows = [{"step": "raw_loaded", "rows_before": n0, "rows_after": n0}]
    def apply(self, frame, step, mask):
        out = frame.loc[mask].copy()
        self.rows.append({"step": step, "rows_before": len(frame), "rows_after": len(out)})
        return out
    def table(self):
        t = pd.DataFrame(self.rows)
        t["rows_removed"] = t["rows_before"] - t["rows_after"]
        t["pct_kept_from_previous"] = (t["rows_after"] / t["rows_before"] * 100).round(2)
        return t

print("Utilities ready.")


Utilities ready.


In [17]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 4 — Load raw data (via fast local copy)
# ═══════════════════════════════════════════════════════════════════════════
if not RAW_PATH.exists():
    raise FileNotFoundError(f"Raw file not found: {RAW_PATH}")

if not LOCAL_RAW_PATH.exists() or LOCAL_RAW_PATH.stat().st_size != RAW_PATH.stat().st_size:
    print("Copying raw file to local Colab storage …")
    shutil.copy2(RAW_PATH, LOCAL_RAW_PATH)
else:
    print("Local raw copy is current.")

cols = list(pd.read_csv(LOCAL_RAW_PATH, nrows=0).columns)
C = {
    "id":           pick_col(cols, ["id"], required=True),
    "title":        pick_col(cols, ["title"], required=True),
    "overview":     pick_col(cols, ["overview"], required=True),
    "release_date": pick_col(cols, ["release_date"], required=True),
    "imdb_id":      pick_col(cols, ["imdb_id"]),
    "status":       pick_col(cols, ["status"]),
    "runtime":      pick_col(cols, ["runtime"]),
    "genres":       pick_col(cols, ["genres", "genres_list"]),
    "cast":         pick_col(cols, ["cast", "Cast_list"]),
    "director":     pick_col(cols, ["director", "Director", "director_name"]),
    "language":     pick_col(cols, ["original_language"]),
    "budget":       pick_col(cols, ["budget"]),
    "revenue":      pick_col(cols, ["revenue"]),
    "popularity":   pick_col(cols, ["popularity"]),
    "vote_average": pick_col(cols, ["vote_average", "imdb_rating"]),
    "vote_count":   pick_col(cols, ["vote_count", "imdb_votes"]),
}
usecols = sorted({c for c in C.values() if c})
raw = pd.read_csv(LOCAL_RAW_PATH, usecols=usecols, low_memory=False)
print(f"Raw loaded: {raw.shape[0]:,} rows × {raw.shape[1]} cols")


Local raw copy is current.
Raw loaded: 1,192,309 rows × 16 cols


In [18]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 5 — Standardise columns
# ═══════════════════════════════════════════════════════════════════════════
def col_or(key, fn, default):
    return fn(raw[C[key]]) if C[key] else default

df = pd.DataFrame(index=raw.index)
df["id"]                = to_num(raw[C["id"]]).astype("Int64")
df["imdb_id"]           = col_or("imdb_id", clean_text, "")
df["title"]             = clean_text(raw[C["title"]])
df["overview"]          = clean_text(raw[C["overview"]])
df["release_date"]      = pd.to_datetime(raw[C["release_date"]], errors="coerce")
df["release_year"]      = df["release_date"].dt.year.astype("Int64")
df["status"]            = col_or("status", clean_text, "")
df["runtime"]           = col_or("runtime", to_num, np.nan)
df["genres_clean"]      = col_or("genres", parse_list_column, "")
df["cast_clean"]        = parse_list_column(raw[C["cast"]], top_n=CFG["cast_top_n"]) if C["cast"] else ""
df["director_name"]     = col_or("director", clean_text, "")
df["original_language"] = col_or("language", clean_text, "")
for k in ["budget", "revenue", "popularity", "vote_average", "vote_count"]:
    df[k] = col_or(k, to_num, np.nan)

placeholders = {"nan": "", "NaN": "", "None": "", "NONE": "", "null": "", "NULL": "",
                "[]": "", "Unknown": "", "N/A": ""}
for c in ["imdb_id", "title", "overview", "status", "genres_clean", "cast_clean",
          "director_name", "original_language"]:
    df[c] = df[c].replace(placeholders)

df["overview_word_count"] = df["overview"].str.split().str.len().fillna(0).astype(int)

floor = CFG["options"]["min_plausible_money"] or 0
df["budget_clean"]  = df["budget"].where(df["budget"] > floor)
df["revenue_clean"] = df["revenue"].where(df["revenue"] > floor)
df["roi_clean"]     = df["revenue_clean"] / df["budget_clean"]
df["log_revenue"]   = np.log1p(df["revenue_clean"])

df["primary_genre"]     = df["genres_clean"].str.split(",").str[0].str.strip()
df["is_tv_movie"]       = df["genres_clean"].str.contains(r"\bTV Movie\b", regex=True, na=False)
df["has_documentary"]   = df["genres_clean"].str.contains(r"\bDocumentary\b", regex=True, na=False)

# Text columns. NB03 embeds `overview` only (TEXT_COLUMN = "overview").
# `text_semantic_plot` (title + genres + plot) is kept for backward compatibility
# but is NOT what the network is built from.
df["text_overview_only"] = df["overview"]
df["text_semantic_plot"] = (df["title"] + ". Genres: " + df["genres_clean"] + ". Plot: "
                            + df["overview"]).str.replace(r"\s+", " ", regex=True).str.strip()

print(f"Standardised: {df.shape}")


Standardised: (1192309, 27)


In [19]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 6 — Filtering funnel
# ═══════════════════════════════════════════════════════════════════════════
snap = pd.Timestamp(CFG["snapshot_date"])
assert snap >= pd.Timestamp(f"{CFG['year_max']}-12-31"), "snapshot_date must not cut into the year window"

f = Funnel(len(df))
w = f.apply(df, "released_status_only",
            df["status"].eq("") | df["status"].str.lower().eq("released"))
w = f.apply(w, f"release_date_on_or_before_{snap.date()}",
            w["release_date"].notna() & (w["release_date"] <= snap))
w = f.apply(w, f"year_window_{CFG['year_min']}_{CFG['year_max']}",
            w["release_year"].between(CFG["year_min"], CFG["year_max"]))
w = f.apply(w, "usable_title", w["title"].str.len() > 0)
w = f.apply(w, "runtime_recorded", w["runtime"].fillna(0) > 0)
w = f.apply(w, f"runtime_{CFG['runtime_min']}_{CFG['runtime_max']}_minutes",
            w["runtime"].between(CFG["runtime_min"], CFG["runtime_max"]))
w = f.apply(w, f"overview_at_least_{CFG['overview_words_thesis']}_words",
            w["overview_word_count"] >= CFG["overview_words_thesis"])
w = f.apply(w, "has_genre", w["genres_clean"].str.len() > 0)
if CFG["exclude_tv_movie"]:
    w = f.apply(w, "exclude_tv_movie_genre", ~w["is_tv_movie"])
if CFG["exclude_documentary_primary"]:
    w = f.apply(w, "exclude_documentary_primary_genre", w["primary_genre"].ne("Documentary"))
if CFG["options"]["exclude_documentary_any_genre"]:
    w = f.apply(w, "exclude_documentary_any_genre", ~w["has_documentary"])
w = f.apply(w, f"minimum_vote_count_{CFG['vote_count_thesis']}",
            w["vote_count"].fillna(0) >= CFG["vote_count_thesis"])

funnel_df = f.table()
display(funnel_df)


,step,rows_before,rows_after,rows_removed,pct_kept_from_previous
0,raw_loaded,1192309,1192309,0,100.00
1,released_status_only,1192309,1170497,21812,98.17
2,release_date_on_or_before_2024-12-31,1170497,994213,176284,84.94
3,year_window_1990_2024,994213,727301,266912,73.15
4,usable_title,727301,727281,20,100.00
5,runtime_recorded,727281,584665,142616,80.39
6,runtime_70_240_minutes,584665,258798,325867,44.26
7,overview_at_least_25_words,258798,200002,58796,77.28
8,has_genre,200002,180239,19763,90.12
9,exclude_tv_movie_genre,180239,170025,10214,94.33


In [20]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 7 — Quality-weighted deduplication (stable row order)
# ═══════════════════════════════════════════════════════════════════════════
w = w.copy()
w["title_key"] = normalize_title(w["title"])
w["title_year_key"] = np.where(
    (w["title_key"].str.len() > 0) & w["release_year"].notna(),
    w["title_key"] + "||" + w["release_year"].astype(str), np.nan)
w["dedupe_quality"] = (
    w["vote_count"].fillna(0).clip(upper=100_000) * 0.001
    + w["popularity"].fillna(0).clip(upper=10_000) * 0.01
    + w["overview_word_count"].fillna(0).clip(upper=200) * 0.05
    + w["revenue_clean"].notna().astype(int) * 10
    + w["budget_clean"].notna().astype(int) * 5
    + w["imdb_id"].str.len().gt(0).astype(int) * 3)

dedup_log = []
def _log(step, before, after):
    dedup_log.append({"step": step, "rows_before": before, "rows_after": after,
                      "rows_removed": before - after})

n = len(w)
has = w["id"].notna()
w = pd.concat([w[has].sort_values(["id", "dedupe_quality"], ascending=[True, False])
                 .drop_duplicates("id", keep="first"), w[~has]], ignore_index=True)
_log("dedupe_exact_tmdb_id", n, len(w))

n = len(w)
has = w["imdb_id"].str.len() > 0
w = pd.concat([w[has].sort_values(["imdb_id", "dedupe_quality"], ascending=[True, False])
                 .drop_duplicates("imdb_id", keep="first"), w[~has]], ignore_index=True)
_log("dedupe_exact_imdb_id", n, len(w))

n = len(w)
ty = w[(w["imdb_id"].str.len() == 0) & w["title_year_key"].notna()] \
       .sort_values(["title_year_key", "dedupe_quality"], ascending=[True, False])
w = pd.concat([w.drop(ty.index), ty.drop_duplicates("title_year_key", keep="first")],
              ignore_index=True)
_log("dedupe_title_year_missing_imdb_only", n, len(w))

dedup_log_df = pd.DataFrame(dedup_log)
display(dedup_log_df)
assert not w.loc[w["id"].notna(), "id"].duplicated().any()
assert not w.loc[w["imdb_id"].str.len() > 0, "imdb_id"].duplicated().any()


,step,rows_before,rows_after,rows_removed
0,dedupe_exact_tmdb_id,98038,98038,0
1,dedupe_exact_imdb_id,98038,98038,0
2,dedupe_title_year_missing_imdb_only,98038,98037,1


In [21]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 8 — Final corpus and focused subsets
# ═══════════════════════════════════════════════════════════════════════════
w = w.rename(columns={"id": "tmdb_id"})

THESIS_COLS = ["tmdb_id", "imdb_id", "title", "original_language", "release_date", "release_year",
               "primary_genre", "genres_clean", "overview", "overview_word_count",
               "text_overview_only", "text_semantic_plot", "runtime", "cast_clean",
               "director_name", "budget_clean", "revenue_clean", "roi_clean", "log_revenue",
               "popularity", "vote_average", "vote_count", "status", "is_tv_movie"]
SEMANTIC_COLS = ["tmdb_id", "imdb_id", "title", "release_year", "primary_genre", "genres_clean",
                 "overview", "overview_word_count", "text_semantic_plot", "runtime", "popularity",
                 "vote_average", "vote_count", "revenue_clean", "log_revenue", "cast_clean",
                 "director_name"]

df_thesis_films = w[THESIS_COLS].copy()

sem_mask = ((df_thesis_films["overview_word_count"] >= CFG["overview_words_semantic"]) &
            (df_thesis_films["vote_count"].fillna(0) >= CFG["vote_count_semantic"]))
df_semantic_analysis  = df_thesis_films.loc[sem_mask, SEMANTIC_COLS].copy()
df_display_candidates = df_semantic_analysis[
    df_semantic_analysis["vote_count"].fillna(0) >= CFG["vote_count_display"]].copy()
df_revenue_analysis   = df_thesis_films[df_thesis_films["revenue_clean"].notna()].copy()
lo, hi = CFG["roi_bounds"]
df_roi_analysis       = df_thesis_films[df_thesis_films["roi_clean"].between(lo, hi, inclusive="neither")].copy()

subset_summary = pd.DataFrame([
    {"dataset": name, "rows": len(d), "cols": d.shape[1]}
    for name, d in [("df_thesis_films", df_thesis_films), ("df_semantic_analysis", df_semantic_analysis),
                    ("df_display_candidates", df_display_candidates),
                    ("df_revenue_analysis", df_revenue_analysis), ("df_roi_analysis", df_roi_analysis)]])
display(subset_summary)


,dataset,rows,cols
0,df_thesis_films,98037,24
1,df_semantic_analysis,20091,17
2,df_display_candidates,13965,17
3,df_revenue_analysis,13186,24
4,df_roi_analysis,7933,24


## Non-story event filter

The thesis compares films by their plots, so recordings that have no plot are excluded: wrestling and MMA event broadcasts, concert films, stand-up specials.

Simple keyword rules match single words such as *vengeance*, *tournament*, *ceremony* and *concert*, and match brand names without word boundaries (`tna` inside "Vie**tna**m"). An audit of the 568 films they flag found 396 real story films among them; removing those would leave systematic gaps in the network: revenge thrillers, Vietnam-war films, tournament films and wedding comedies.

The filter therefore works in two steps. The keyword rules flag candidates, and a candidate stays out only if it is a recording without a plot. A *film about* a tournament (*A Knight's Tale*, *Warrior*) stays in; a *filmed* tournament (*WWE SummerSlam 2019*) is removed. Every flagged film is saved with its decision in `corpus_audit/event_filter_audit.csv`.

In [22]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 9 — Non-story event filter → df_network_corpus
# ═══════════════════════════════════════════════════════════════════════════
# Step 1: keyword rules flag candidates.
# Step 2: the strict rule keeps a candidate out ONLY if it is a recording without a plot.
#         Step 2 only acts on step-1 candidates, so it can never remove a film the keyword
#         rules keep.
Q = df_semantic_analysis.copy()
Q["overview"] = Q["overview"].fillna("").astype(str).str.strip()
Q = Q[Q["overview"].str.len() > 0]
title_l  = Q["title"].fillna("").astype(str).str.lower()
over_l   = Q["overview"].str.lower()
genres   = Q["genres_clean"].fillna("").astype(str)
combined = title_l + " " + over_l + " " + genres.str.lower()

# ── Step 1: keyword rules ──────────────────────────────────────────────────
LEGACY = {
    "brand": r"\b(wwe|wwf|wcw|nxt|aew|tna|impact wrestling|wrestlemania|royal rumble|summerslam|"
             r"survivor series|elimination chamber|money in the bank|hell in a cell|clash of champions|"
             r"crown jewel|no mercy|bad blood|backlash|armageddon|judgment day|king of the ring|"
             r"night of champions|ufc|bellator|one championship|pride fc|k-1|glory kickboxing)\b",
    "language": r"\b(pay[-\s]?per[-\s]?view|main event|championship match|title match|cage match|"
                r"ladder match|tag team|wrestling event|professional wrestling|mixed martial arts|"
                r"martial arts event|boxing match|boxing event|fight night|live event|live broadcast|"
                r"broadcast live|sports event|tournament|final match)\b",
    "performance": r"\b(concert|live at|live from|live in concert|stand[-\s]?up|standup special|"
                   r"comedy special|awards show|award ceremony|ceremony|pageant|stage production|"
                   r"theatre production|theater production|opera performance|ballet performance)\b",
    "template": r"\b(features matches|features performances|superstars compete|fighters compete|"
                r"teams compete|takes place at|held at|emanates from|annual event|pay-per-view event)\b",
}
LEGACY_HARD = (r"wwe|wwf|wcw|nxt|aew|tna|impact wrestling|wrestlemania|royal rumble|summerslam|survivor series|"
    r"elimination chamber|money in the bank|hell in a cell|clash of champions|crown jewel|no mercy|"
    r"bad blood|backlash|armageddon|judgment day|king of the ring|night of champions|"
    r"great american bash|battleground|extreme rules|fastlane|roadblock|vengeance|fully loaded|"
    r"in your house|taboo tuesday|december to dismember|starrcade|slamboree|halloween havoc|"
    r"bash at the beach|souled out|superbrawl|uncensored|fall brawl|spring stampede|"
    r"ufc|bellator|one championship|pride fc|k-1|glory kickboxing")   # no word boundaries: 'tna' hits 'vietnam'

lf = pd.DataFrame({k: combined.str.contains(p, regex=True, na=False) for k, p in LEGACY.items()}, index=Q.index)
eventish_genre = genres.str.contains(r"Action|TV Movie|Music", case=False, regex=True)
legacy_rule = pd.Series("", index=Q.index)
legacy_rule[lf["brand"]] = "brand"
legacy_rule[(legacy_rule == "") & lf["performance"]] = "performance"
legacy_rule[(legacy_rule == "") & lf["language"] & lf["template"]] = "language+template"
legacy_rule[(legacy_rule == "") & lf["language"] & eventish_genre] = "language+genre"
hard = title_l.str.contains(LEGACY_HARD, regex=True) | over_l.str.contains(LEGACY_HARD, regex=True)
legacy_rule[(legacy_rule == "") & hard] = "hard_pattern"

# ── Step 2: strict rule — recordings without a plot ────────────────────────
BRAND = (r"\b(?:wwe|wwf|wcw|nxt|aew|tna|impact wrestling|wrestlemania|royal rumble|summerslam|survivor series|"
         r"elimination chamber|hell in a cell|clash of champions|king of the ring|night of champions|"
         r"great american bash|extreme rules|taboo tuesday|december to dismember|starrcade|slamboree|"
         r"halloween havoc|bash at the beach|superbrawl|fall brawl|spring stampede|ufc|bellator|"
         r"one championship|pride fc|glory kickboxing)\b")
EVENT_WORDING = (r"\b(?:features matches|features performances|superstars compete|fighters compete|teams compete|"
                 r"emanates from|annual event|pay[-\s]?per[-\s]?view|main event|card features|live event)\b")
PERFORMANCE = (r"\b(?:concert|live at|live from|live in concert|stand[-\s]?up|standup special|comedy special|"
               r"awards show|award ceremony|stage production|theatre production|theater production|"
               r"opera performance|ballet performance)\b")
REC_TITLE    = r"\b(?:in concert|live from|live at|live in|unplugged)\b"
REC_OVERVIEW = (r"\b(?:this concert|concert film|recorded live|filmed live|live recording|this documentary|"
                r"stand-up special|comedy special)\b")
NARRATIVE    = (r"Drama|Romance|Thriller|Crime|Horror|Science Fiction|Fantasy|Family|Adventure|Action|"
                r"Mystery|War|Western|Animation|History")
STORY_GENRE  = r"Animation|Family|Mystery|Science Fiction|Fantasy|Horror"   # e.g. Scooby-Doo! WrestleMania Mystery

text = title_l + " " + over_l
is_doc    = genres.str.contains("Documentary")
narrative = genres.str.contains(NARRATIVE, regex=True)
story     = genres.str.contains(STORY_GENRE, regex=True)
reason = pd.Series("", index=Q.index)
reason = reason.mask(title_l.str.contains(BRAND, regex=True) & ~story, "wrestling/MMA brand in title")
reason = reason.mask((reason == "") & text.str.contains(BRAND, regex=True) & text.str.contains(EVENT_WORDING, regex=True),
                     "brand + event wording")
reason = reason.mask((reason == "") & text.str.contains(BRAND, regex=True) & is_doc, "wrestling/MMA documentary")
reason = reason.mask((reason == "") & title_l.str.contains(REC_TITLE, regex=True), "recorded performance (title)")
reason = reason.mask((reason == "") & over_l.str.contains(REC_OVERVIEW, regex=True) & (is_doc | ~narrative),
                     "recorded performance (overview)")
reason = reason.mask((reason == "") & text.str.contains(PERFORMANCE, regex=True) & ~narrative,
                     "performance, no narrative genre")

legacy_out = legacy_rule != ""
removed = (legacy_out & (reason != "")) if CFG["event_filter"] == "strict" else legacy_out

quarantine_audit = Q.loc[legacy_out, ["tmdb_id", "title", "release_year", "primary_genre",
                                      "genres_clean", "vote_count", "overview"]].copy()
quarantine_audit.insert(0, "legacy_rule", legacy_rule[legacy_out])
quarantine_audit.insert(1, "strict_reason", reason[legacy_out])
quarantine_audit.insert(2, "decision", np.where(removed[legacy_out], "removed", "kept (has a story)"))
quarantine_audit = quarantine_audit.sort_values(["decision", "vote_count"], ascending=[False, False])

df_network_corpus = Q.loc[~removed].reset_index(drop=True)
expected = CFG["expected_network_rows"][CFG["event_filter"]]

print(f"Event filter mode:         {CFG['event_filter']}")
print(f"Semantic corpus:           {len(Q):,}")
print(f"Flagged by keyword rules:  {legacy_out.sum():,}")
print(f"Removed (no plot):         {removed.sum():,}")
print(f"Kept after review:         {(legacy_out & ~removed).sum():,}")
print(f"Network corpus:            {len(df_network_corpus):,}  (expected {expected:,})")
display(quarantine_audit.groupby(["decision", "strict_reason"]).size().to_frame("films"))
print("\nHighest-profile films kept that keyword rules alone would remove:")
display(quarantine_audit[quarantine_audit["decision"].str.startswith("kept")]
        [["legacy_rule", "title", "release_year", "genres_clean", "vote_count"]].head(15))


Event filter mode:         strict
Semantic corpus:           20,091
Flagged by legacy rules:   568
Removed (no plot):         172
Kept after review:         396
Network corpus:            19,919  (expected 19,919)


films
decision           strict_reason                         
kept (has a story)                                    396
removed            brand + event wording                1
                   performance, no narrative genre     62
                   recorded performance (overview)     30
                   recorded performance (title)        19
                   wrestling/MMA brand in title        59
                   wrestling/MMA documentary            1


Highest-profile films kept that the legacy filter removed:


,legacy_rule,title,release_year,genres_clean,vote_count
1518,brand,Terminator 2: Judgment Day,1991,"Action, Thriller, Science Fiction",14176.0
22845,hard_pattern,The Equalizer,2014,"Thriller, Action, Crime",9896.0
26391,hard_pattern,Godzilla,2014,"Action, Drama, Science Fiction",9726.0
7075,brand,Armageddon,1998,"Action, Thriller, Science Fiction, Adventure",8522.0
30749,hard_pattern,Thor: Love and Thunder,2022,"Fantasy, Action, Comedy",8514.0
9075,performance,Hellboy,2004,"Fantasy, Action",7366.0
9622,brand,Terminator 3: Rise of the Machines,2003,"Action, Thriller, Science Fiction",7201.0
4510,hard_pattern,Die Hard: With a Vengeance,1995,"Action, Thriller",6621.0
25234,hard_pattern,American Gangster,2007,"Drama, Crime",5998.0
83119,brand,Terminator: Dark Fate,2019,"Science Fiction, Action, Adventure, Thriller",5625.0


## Diagnostics

Nothing in this section changes the data. It quantifies sample decisions, each of which can be changed with an option in `CFG`.


In [23]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 10 — Diagnostics
# ═══════════════════════════════════════════════════════════════════════════
diag = {}

# (a) Cast parser: how many final-corpus rows would a naive comma split corrupt?
if C["cast"]:
    raw_cast = raw.loc[raw[C["id"]].isin(df_thesis_films["tmdb_id"].astype("int64")),
                       [C["id"], C["cast"]]].drop_duplicates(C["id"]).set_index(C["id"])[C["cast"]]
    old = raw_cast.map(lambda x: legacy_parse(x, CFG["cast_top_n"]))
    new = parse_list_column(raw_cast, top_n=CFG["cast_top_n"])
    changed = old.ne(new)
    in_net = raw_cast.index.isin(df_network_corpus["tmdb_id"].astype("int64"))
    diag["cast_rows_corrected_thesis"]  = int(changed.sum())
    diag["cast_rows_corrected_network"] = int((changed & in_net).sum())
    cast_fix_examples = pd.DataFrame({"naive_split": old[changed], "parsed": new[changed]}).head(15)

# (b) Documentaries kept because Documentary is a secondary genre
doc_secondary = df_network_corpus["genres_clean"].str.contains(r"\bDocumentary\b", regex=True, na=False)
diag["network_films_with_documentary_secondary_genre"] = int(doc_secondary.sum())

# (c) Implausible money values (TMDB often stores "12" meaning $12M)
tiny_budget  = df_thesis_films["budget_clean"].lt(10_000)
tiny_revenue = df_thesis_films["revenue_clean"].lt(10_000)
diag["thesis_films_budget_below_10k"]  = int(tiny_budget.sum())
diag["thesis_films_revenue_below_10k"] = int(tiny_revenue.sum())
diag["roi_films_with_budget_or_revenue_below_10k"] = int(
    (df_roi_analysis["budget_clean"].lt(10_000) | df_roi_analysis["revenue_clean"].lt(10_000)).sum())

# (d) Overviews unlikely to be English (all-MiniLM-L6-v2 is an English model)
letters = df_network_corpus["overview"].str.count(r"[A-Za-z]")
non_space = df_network_corpus["overview"].str.count(r"\S").clip(lower=1)
likely_non_english = (letters / non_space) < 0.6
diag["network_overviews_likely_non_english"] = int(likely_non_english.sum())
diag["network_films_non_en_original_language_pct"] = round(
    float(df_network_corpus.merge(df_thesis_films[["tmdb_id", "original_language"]], on="tmdb_id")
          ["original_language"].ne("en").mean() * 100), 1)

# (e) Event filter: borderline decisions worth a glance
kept_non_narrative = (quarantine_audit["decision"].str.startswith("kept") &
                      ~quarantine_audit["genres_clean"].str.contains(NARRATIVE, regex=True, na=False))
removed_story_genre = (quarantine_audit["decision"].eq("removed") &
                       quarantine_audit["genres_clean"].str.contains(STORY_GENRE, regex=True, na=False))
qa_review = quarantine_audit[kept_non_narrative | removed_story_genre]
diag["event_filter_removed"] = int(removed.sum())
diag["event_filter_kept_after_review"] = int((legacy_out & ~removed).sum())
diag["event_filter_borderline_to_glance_at"] = len(qa_review)

diag_df = pd.DataFrame([{"diagnostic": k, "value": v} for k, v in diag.items()])
display(diag_df)
if C["cast"] and len(cast_fix_examples):
    print("\nCast parser — examples of names a naive split would break:"); display(cast_fix_examples)
print("\nBorderline event-filter decisions (kept with no narrative genre, or removed despite a story genre):")
display(qa_review[["decision", "strict_reason", "title", "release_year", "genres_clean"]].head(20))


,diagnostic,value
0,cast_rows_corrected_thesis,2955.0
1,cast_rows_corrected_network,1051.0
2,network_films_with_documentary_secondary_genre,138.0
3,thesis_films_budget_below_10k,1147.0
4,thesis_films_revenue_below_10k,643.0
5,roi_films_with_budget_or_revenue_below_10k,323.0
6,network_overviews_likely_non_english,0.0
7,network_films_non_en_original_language_pct,46.9
8,event_filter_removed,172.0
9,event_filter_kept_after_review,396.0



Cast parser fix — examples:


,v1_cast,v2_cast
id,,
13,"Alessandro, Danté McCarthy, Tom Hanks, John William Galt, Jim Damron, Sonny Shroyer, Zach Hanner, George Kelly, Jaso...","John Randall, Steven Griffith, Juan Singleton, Charles Boswell, Daniel C. Striepeke"
58,"Brien, Dermot Keaney, Jonathan Linsley, Andy Beckwith, Mike Haberecht, Mackenzie Crook, David Schofield, Neil Panlas...","Spider Madison, Kevin McNally, Georges Trillat, Nej Adamson, Simon Meacock"
63,"Chip, Neill, Crispian Sallis, Christopher Plummer, Harry O","Bruce Kirkpatrick, Michael Chance, Felix Pire, Nell Johnson, Barry Price"
65,"Ox, Angelo Wilson, Venicia Foreman, Taryn Manning, Day Golfin, DJ Head, Brittany Murphy, Obie Trice, Melissa Zaglani...","Xzibit, Miz-Korona, Eminem, Mekhi Phifer, Bushman"
70,"Ace, Angerio, Don Familton, Sunshine Chantal Parkman, Ned Eisenberg, Jason Williams, Vincent Foster, Ray Corona, Kim...","Roy Nugent, Riki Lindhome, Michael Peña, Kirsten Berman, Erica Grant"
73,"Big Dawg, Angelo, Richard Noyce, Sam Vlahos, Tara Blanchard, Robert","Nigel Miguel, David Basulto, Darrell Britt, Fairuza Balk, Nicholas R. Oleson"
74,"Ambrosio, Vladislav Kozlov, John Scurti, Bruce W. Derdoski Jr., Cass Asher, Ann Robinson, Dakota Fanning, Kyle S. Br...","Art Chudabala, John Michael Bolger, Tanda Kerín, Jim Hanna, Amy Ryan"
75,Gingir,"Jack Nicholson, Janice Rivera, Betty Bunch, Rod Steiger, Vinny Argiro"
76,Andrla,"Wilbirg Reiter, Adam Goldberg, Wolfgang Glüxam, Alexandra Seibel, Peter Ily Huemer"



Borderline event-filter decisions (kept with no narrative genre, or removed despite a story genre):


,decision,strict_reason,title,release_year,genres_clean
83313,kept (has a story),,Fighting with My Family,2019,Comedy
8603,kept (has a story),,Drop Dead Gorgeous,1999,Comedy
81784,kept (has a story),,Thi Mai,2018,Comedy
17488,kept (has a story),,Os Normais: O Filme,2003,Comedy
83911,kept (has a story),,Overnight Republic,2017,Comedy
59573,kept (has a story),,Swearnet: The Movie,2014,Comedy
75093,kept (has a story),,Niyazi Gül Dörtnala,2015,Comedy
14550,kept (has a story),,Erkan & Stefan 2,2002,Comedy


In [24]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 11 — Integrity checks
# ═══════════════════════════════════════════════════════════════════════════
ids = lambda d: set(d["tmdb_id"].astype("int64"))
checks = {
    "no duplicate tmdb_id":              not df_thesis_films["tmdb_id"].duplicated().any(),
    "years inside window":               df_thesis_films["release_year"].between(CFG["year_min"], CFG["year_max"]).all(),
    "semantic ⊂ thesis":                 ids(df_semantic_analysis) <= ids(df_thesis_films),
    "network ⊂ semantic":                ids(df_network_corpus) <= ids(df_semantic_analysis),
    "display ⊂ semantic":                ids(df_display_candidates) <= ids(df_semantic_analysis),
    "roi ⊂ revenue":                     ids(df_roi_analysis) <= ids(df_revenue_analysis),
    "no empty network overviews":        df_network_corpus["overview"].str.len().gt(0).all(),
    "no wrestling/MMA broadcasts left":  not (df_network_corpus["title"].str.lower().str.contains(BRAND, regex=True, na=False)
                                              & ~df_network_corpus["genres_clean"].str.contains(STORY_GENRE, regex=True, na=False)).any(),
    "network size as expected":          len(df_network_corpus) == expected,
}
check_df = pd.DataFrame([{"check": k, "passed": bool(v)} for k, v in checks.items()])
display(check_df)
if not check_df["passed"].all():
    raise AssertionError("Integrity checks failed — see table above.")


,check,passed
0,no duplicate tmdb_id,True
1,years inside window,True
2,semantic ⊂ thesis,True
3,network ⊂ semantic,True
4,display ⊂ semantic,True
5,roi ⊂ revenue,True
6,no empty network overviews,True
7,no wrestling/MMA broadcasts left,True
8,network size as expected,True


## Safety guard

The communities, manual labels and every brick depend on NB03 having embedded **these exact films in this exact order**. This cell compares the new corpus with:
1. the files already on Drive, and
2. the embedding index NB03 saved (`sbert_minilm_embedding_index.csv`).

If the film list or order differs and `allow_corpus_change` is `False`, nothing is written. Content-only differences (such as the corrected `cast_clean`) are reported but allowed.


In [25]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 12 — Safety guard
# ═══════════════════════════════════════════════════════════════════════════
OUTPUTS = {
    "df_thesis_films":       (df_thesis_films,       CORE_DIR / "df_thesis_films.csv"),
    "df_semantic_analysis":  (df_semantic_analysis,  CORE_DIR / "df_semantic_analysis.csv"),
    "df_network_corpus":     (df_network_corpus,     CORE_DIR / "df_network_corpus.csv"),
    "df_display_candidates": (df_display_candidates, CORE_DIR / "df_display_candidates.csv"),
    "df_revenue_analysis":   (df_revenue_analysis,   CORE_DIR / "df_revenue_analysis.csv"),
    "df_roi_analysis":       (df_roi_analysis,       CORE_DIR / "df_roi_analysis.csv"),
}

guard_rows = []
for name, (new_df, path) in OUTPUTS.items():
    row = {"dataset": name, "new_rows": len(new_df), "existing_rows": None,
           "same_films_same_order": None, "columns_with_changed_values": ""}
    if path.exists():
        old_df = pd.read_csv(path, low_memory=False)
        row["existing_rows"] = len(old_df)
        same = (len(old_df) == len(new_df) and
                old_df["tmdb_id"].astype("int64").tolist() == new_df["tmdb_id"].astype("int64").tolist())
        row["same_films_same_order"] = same
        if same:
            changed = []
            def _norm(s):   # empty string, NaN, <NA>, NaT all mean "missing" once saved to CSV
                s = s.reset_index(drop=True).astype(object)
                return s.where(s.notna(), "").astype(str).replace({"nan": "", "<NA>": "", "NaT": "", "None": ""})
            for c in new_df.columns.intersection(old_df.columns):
                a, b = _norm(new_df[c]), _norm(old_df[c])
                if c == "release_date":
                    a = pd.to_datetime(new_df[c]).dt.strftime("%Y-%m-%d").reset_index(drop=True)
                    b = pd.to_datetime(old_df[c], errors="coerce").dt.strftime("%Y-%m-%d")
                n_diff = int((a != b).sum())
                if n_diff and new_df[c].dtype.kind in "fi":
                    n_diff = int((~np.isclose(pd.to_numeric(new_df[c], errors="coerce").values,
                                               pd.to_numeric(old_df[c], errors="coerce").values,
                                               equal_nan=True)).sum())
                if n_diff:
                    changed.append(f"{c} ({n_diff:,})")
            row["columns_with_changed_values"] = ", ".join(changed)
    guard_rows.append(row)

# Strongest check: does the network corpus match what NB03 actually embedded?
if EMBED_INDEX_FILE.exists():
    emb = pd.read_csv(EMBED_INDEX_FILE)
    matches_embeddings = emb["tmdb_id"].astype("int64").tolist() == \
                         df_network_corpus["tmdb_id"].astype("int64").tolist()
else:
    matches_embeddings = None

guard_df = pd.DataFrame(guard_rows)
display(guard_df)
print("Network corpus matches NB03 embedding index:", matches_embeddings)

order_broken = guard_df["same_films_same_order"].eq(False).any() or matches_embeddings is False
SAFE_TO_WRITE = (not order_broken) or CFG["allow_corpus_change"]

if order_broken and not CFG["allow_corpus_change"]:
    print("\n⛔ The film list or order would change. Nothing will be written.\n"
          "   The network, communities and manual labels would no longer line up.\n"
          "   If this change is intended, set CFG['allow_corpus_change'] = True and plan to\n"
          "   re-run NB03 → NB05 and relabel communities.")
elif order_broken:
    print("\n⚠️  Corpus WILL change (allow_corpus_change=True).\n"
          "   Expected: df_network_corpus becomes 19,919 films because of the corrected event filter.\n"
          "   Next: rebuild the network in NB03, then set allow_corpus_change back to False.")
else:
    print("\n✅ Same films in the same order. Safe to write.")


,dataset,new_rows,existing_rows,same_films_same_order,columns_with_changed_values
0,df_thesis_films,98037,98037,True,
1,df_semantic_analysis,20091,20091,True,
2,df_network_corpus,19919,19919,True,
3,df_display_candidates,13965,13965,True,
4,df_revenue_analysis,13186,13186,True,
5,df_roi_analysis,7933,7933,True,


Network corpus matches NB03 embedding index: True

✅ Same films in the same order. Safe to write.


In [26]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 13 — Export data, audit tables and manifest
# ═══════════════════════════════════════════════════════════════════════════
if not SAFE_TO_WRITE:
    raise SystemExit("Export skipped by safety guard (see previous cell).")

for name, (d, path) in OUTPUTS.items():
    d.to_csv(path, index=False)
    print(f"  ✓ {path.name:32s} {len(d):>7,} rows")

year_counts  = df_thesis_films["release_year"].value_counts().sort_index() \
                 .rename_axis("release_year").reset_index(name="movie_count")
genre_counts = df_thesis_films["primary_genre"].replace("", np.nan).dropna().value_counts().head(30) \
                 .rename_axis("primary_genre").reset_index(name="movie_count")

AUDIT_TABLES = {
    TABLES_DIR / "filtering_funnel.csv":               funnel_df,    
    TABLES_DIR / "dedup_log.csv":                      dedup_log_df,
    TABLES_DIR / "movies_per_year_thesis_corpus.csv":  year_counts,
    TABLES_DIR / "primary_genre_counts.csv":           genre_counts,
    AUDIT_DIR / "event_filter_audit.csv":              quarantine_audit,
    AUDIT_DIR / "event_filter_borderline.csv":         qa_review,
    AUDIT_DIR / "corpus_diagnostics.csv":              diag_df,
    AUDIT_DIR / "integrity_checks.csv":                check_df,
    AUDIT_DIR / "safety_guard_report.csv":             guard_df,
}
for path, d in AUDIT_TABLES.items():
    d.to_csv(path, index=False)
    print(f"  ✓ {path.relative_to(TABLES_DIR)}")

manifest = {
    "notebook": "01_build_curated_thesis_corpus_v2",
    "built_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "config": {k: v for k, v in CFG.items() if k != "project_dir"},
    "row_counts": {name: len(d) for name, (d, _) in OUTPUTS.items()},
    "fingerprints": {name: fingerprint(d["tmdb_id"]) for name, (d, _) in OUTPUTS.items()},
    "network_corpus_matches_embedding_index": matches_embeddings,
    "embedded_text_column_in_nb03": "overview",
    "event_filter": CFG["event_filter"],
}
(CORE_DIR / "corpus_manifest.json").write_text(json.dumps(manifest, indent=2, default=str))
print("  ✓ corpus_manifest.json")


  ✓ df_thesis_films.csv               98,037 rows
  ✓ df_semantic_analysis.csv          20,091 rows
  ✓ df_network_corpus.csv             19,919 rows
  ✓ df_display_candidates.csv         13,965 rows
  ✓ df_revenue_analysis.csv           13,186 rows
  ✓ df_roi_analysis.csv                7,933 rows
  ✓ filtering_funnel.csv
  ✓ dedup_log.csv
  ✓ movies_per_year_thesis_corpus.csv
  ✓ primary_genre_counts.csv
  ✓ corpus_audit/event_filter_audit.csv
  ✓ corpus_audit/event_filter_borderline.csv
  ✓ corpus_audit/corpus_diagnostics.csv
  ✓ corpus_audit/integrity_checks.csv
  ✓ corpus_audit/safety_guard_report.csv
  ✓ corpus_manifest.json


## Outputs

* `processed/core/df_network_corpus.csv` (19,919 films) is the input of Notebook 03, which checks it against `corpus_manifest.json` → `fingerprints.df_network_corpus`.
* `corpus_audit/event_filter_audit.csv` lists every film the event filter flagged, with its decision; `corpus_audit/event_filter_borderline.csv` lists the edge cases.
* Recordings without a plot (wrestling and MMA event broadcasts, concert films and stand-up specials; n = 172) are excluded because the network compares films by their plot descriptions. The network is built from the plot overviews only, not from titles or genre labels.